# 03 — Pré-processamento dos Dados Meteorológicos

Este notebook tem como objetivo preparar a base meteorológica para a etapa de modelagem preditiva.

A base já foi convertida de CSV para Parquet no notebook anterior. Aqui serão feitas etapas de limpeza, padronização, tratamento de valores inválidos, remoção de outliers, imputação de valores ausentes e geração de uma base final para os modelos de aprendizado de máquina.

A variável-alvo considerada será a temperatura do ar.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, when, count, avg, min, max, stddev, round,
    regexp_replace, regexp_extract, to_date, year, month, dayofmonth
)
import builtins
import unicodedata
import re

## 1. Inicialização da SparkSession

A `SparkSession` é o ponto de entrada para utilizar o Spark. Ela permite carregar dados, criar DataFrames, aplicar transformações e executar operações distribuídas.

In [ ]:
spark = (
    SparkSession.builder
    .appName("Preprocessamento_Weather_SP")
    .getOrCreate()
)

spark

## 2. Carregamento da base em Parquet

A base utilizada neste notebook vem do arquivo Parquet gerado no notebook de conversão. O formato Parquet é mais eficiente que CSV para leitura e processamento em ambientes de Big Data, pois armazena os dados em formato colunar.

In [ ]:
input_path = "/home/jovyan/work/data/processed/weather_sp_parquet"

df = spark.read.parquet(input_path)

df.show(5, truncate=False)
df.printSchema()

## 3. Normalização dos nomes das colunas

Os nomes originais das colunas possuem acentos, espaços, vírgulas, parênteses e outros caracteres especiais. Isso pode dificultar o uso das colunas em expressões Spark.

Nesta etapa, os nomes são padronizados para letras minúsculas, sem acentos e com separação por `_`.

In [ ]:
def normalizar_nome_coluna(nome):
    nome = nome.strip()
    nome = unicodedata.normalize("NFKD", nome)
    nome = nome.encode("ASCII", "ignore").decode("utf-8")
    nome = nome.lower()
    nome = re.sub(r"[^a-z0-9]+", "_", nome)
    nome = re.sub(r"_+", "_", nome)
    nome = nome.strip("_")
    return nome

colunas_normalizadas = [normalizar_nome_coluna(c) for c in df.columns]

df = df.toDF(*colunas_normalizadas)

df.columns

## 4. Renomeação das variáveis principais

Após a normalização automática, algumas colunas ainda ficam com nomes longos. Nesta etapa, as principais variáveis meteorológicas são renomeadas para nomes mais simples e legíveis.

Exemplo: a coluna de temperatura do ar passa a ser chamada apenas de `temperatura`.

In [ ]:
df = (
    df
    .withColumnRenamed("data", "data")
    .withColumnRenamed("hora", "hora")
    .withColumnRenamed("precipitacao_total_horario_mm", "precipitacao")
    .withColumnRenamed("pressao_atmosferica_ao_nivel_da_estacao_horaria_mb", "pressao")
    .withColumnRenamed("pressao_atmosferica_max_na_hora_ant_aut_mb", "pressao_maxima")
    .withColumnRenamed("pressao_atmosferica_min_na_hora_ant_aut_mb", "pressao_minima")
    .withColumnRenamed("radiacao_global_kj_m2", "radiacao")
    .withColumnRenamed("temperatura_do_ar_bulbo_seco_horaria_c", "temperatura")
    .withColumnRenamed("umidade_rel_max_na_hora_ant_aut", "umidade_maxima")
    .withColumnRenamed("umidade_rel_min_na_hora_ant_aut", "umidade_minima")
    .withColumnRenamed("umidade_relativa_do_ar_horaria", "umidade")
    .withColumnRenamed("vento_direcao_horaria_gr_gr", "direcao_vento")
    .withColumnRenamed("vento_rajada_maxima_m_s", "rajada_vento")
    .withColumnRenamed("vento_velocidade_horaria_m_s", "velocidade_vento")
)

df.printSchema()

## 5. Seleção das colunas relevantes

Nesta etapa são mantidas apenas as colunas úteis para a análise e futura modelagem.

Foram selecionadas variáveis temporais, geográficas e meteorológicas. Colunas irrelevantes ou redundantes para o objetivo inicial são descartadas.

In [ ]:
colunas_uteis = [
    "data", "hora", "region", "state", "station", "station_code",
    "latitude", "longitude", "height",
    "temperatura", "umidade", "umidade_maxima", "umidade_minima",
    "pressao", "pressao_maxima", "pressao_minima",
    "precipitacao", "radiacao", "velocidade_vento", "rajada_vento", "direcao_vento"
]

colunas_uteis = [c for c in colunas_uteis if c in df.columns]

df = df.select(*colunas_uteis)

df.show(5, truncate=False)
df.printSchema()

## 6. Conversão das colunas numéricas

Algumas colunas numéricas podem ser interpretadas como texto, especialmente quando vêm de arquivos CSV.

Nesta etapa, as variáveis meteorológicas e geográficas são convertidas para o tipo `double`, garantindo que possam ser usadas em cálculos estatísticos e modelos de aprendizado de máquina.

In [ ]:
colunas_numericas = [
    "latitude", "longitude", "height",
    "temperatura", "umidade", "umidade_maxima", "umidade_minima",
    "pressao", "pressao_maxima", "pressao_minima",
    "precipitacao", "radiacao", "velocidade_vento", "rajada_vento", "direcao_vento"
]

colunas_numericas = [c for c in colunas_numericas if c in df.columns]

for c in colunas_numericas:
    df = df.withColumn(
        c,
        regexp_replace(col(c).cast("string"), ",", ".").cast("double")
    )

df.printSchema()

## 7. Criação de variáveis temporais

A data e a hora são transformadas em variáveis numéricas úteis para os modelos.

São criadas as colunas:

- `ano`
- `mes`
- `dia`
- `hora_num`

Essas variáveis permitem que os modelos capturem padrões temporais, como variações sazonais e mudanças ao longo dos anos.

In [ ]:
df = df.withColumn("data_formatada", to_date(col("data")))

df = (
    df
    .withColumn("ano", year(col("data_formatada")))
    .withColumn("mes", month(col("data_formatada")))
    .withColumn("dia", dayofmonth(col("data_formatada")))
    .withColumn("hora_num", regexp_extract(col("hora").cast("string"), r"(\d{2})", 1).cast("int"))
)

df.select("data", "data_formatada", "ano", "mes", "dia", "hora", "hora_num").show(20, truncate=False)

### 7.1 Conversão alternativa de data

Caso a conversão anterior gere valores nulos em `ano`, `mes` ou `dia`, utilize a célula abaixo. Ela considera o formato brasileiro de data: `dd/MM/yyyy`.

Execute esta célula apenas se a anterior não funcionar corretamente.

In [ ]:
# df = df.withColumn("data_formatada", to_date(col("data"), "dd/MM/yyyy"))
# df = (
#     df
#     .withColumn("ano", year(col("data_formatada")))
#     .withColumn("mes", month(col("data_formatada")))
#     .withColumn("dia", dayofmonth(col("data_formatada")))
#     .withColumn("hora_num", regexp_extract(col("hora").cast("string"), r"(\d{2})", 1).cast("int"))
# )
# df.select("data", "data_formatada", "ano", "mes", "dia", "hora", "hora_num").show(20, truncate=False)
print("Célula alternativa desativada — ative removendo os '#' se necessário.")

## 8. Verificação do tamanho inicial da base

Antes dos tratamentos, é importante registrar a quantidade inicial de linhas e colunas. Isso permite comparar o impacto das etapas de limpeza realizadas posteriormente.

In [ ]:
linhas_iniciais = df.count()
print(f"Linhas iniciais: {linhas_iniciais:,}")
print(f"Colunas iniciais: {len(df.columns)}")

## 9. Verificação e remoção de registros duplicados

Nesta etapa é verificado se existem registros duplicados considerando a estação meteorológica, a data e a hora da medição.

Como os dados são horários, espera-se que cada estação tenha apenas um registro por horário. Portanto, a combinação `station_code`, `data` e `hora` é utilizada como chave para identificar duplicidades.

In [ ]:
colunas_chave = [c for c in ["station_code", "data", "hora"] if c in df.columns]

total_antes = df.count()
df = df.dropDuplicates(colunas_chave)
total_depois = df.count()

print(f"Antes: {total_antes:,} | Depois: {total_depois:,} | Removidas: {total_antes - total_depois:,}")

## 10. Remoção de registros sem variável-alvo

A variável-alvo do projeto é a temperatura do ar. Registros sem valor de temperatura não podem ser usados para treinar modelos supervisionados.

Por isso, linhas com `temperatura` nula são removidas.

In [ ]:
df = df.filter(col("temperatura").isNotNull())

print("Linhas após remover temperatura nula:", df.count())

## 11. Tratamento de valores sentinela

Alguns datasets meteorológicos usam valores extremamente negativos, como `-9999`, para indicar ausência de medição.

Nesta etapa, valores menores ou iguais a `-999` são convertidos para nulo.

In [ ]:
for c in colunas_numericas:
    df = df.withColumn(
        c,
        when(col(c) <= -999, None).otherwise(col(c))
    )

df.show(5, truncate=False)

## 12. Remoção de valores fisicamente inválidos

Nesta etapa são removidos valores incompatíveis com a realidade física das variáveis meteorológicas.

Exemplos:

- Temperatura fora de uma faixa aceitável
- Umidade menor que 0 ou maior que 100
- Precipitação negativa
- Velocidade do vento negativa

Valores ausentes em variáveis explicativas são mantidos, pois serão tratados posteriormente por imputação.

In [ ]:
df_tratado = df

if "temperatura" in df_tratado.columns:
    df_tratado = df_tratado.filter((col("temperatura") >= -10) & (col("temperatura") <= 50))

for col_umidade in ["umidade", "umidade_maxima", "umidade_minima"]:
    if col_umidade in df_tratado.columns:
        df_tratado = df_tratado.filter(
            col(col_umidade).isNull() | ((col(col_umidade) >= 0) & (col(col_umidade) <= 100))
        )

for col_pos in ["precipitacao", "velocidade_vento", "rajada_vento"]:
    if col_pos in df_tratado.columns:
        df_tratado = df_tratado.filter(col(col_pos).isNull() | (col(col_pos) >= 0))

print(f"Antes dos filtros físicos: {df.count():,}")
print(f"Após filtros físicos     : {df_tratado.count():,}")

## 13. Remoção de outliers da temperatura por IQR

A técnica IQR utiliza o intervalo interquartil para identificar valores muito afastados da distribuição central dos dados.

São calculados:

- Q1: primeiro quartil
- Q3: terceiro quartil
- IQR: diferença entre Q3 e Q1
- Limite inferior: Q1 - 1,5 × IQR
- Limite superior: Q3 + 1,5 × IQR

Registros com temperatura fora desses limites são removidos.

In [ ]:
q1, q3 = df_tratado.approxQuantile("temperatura", [0.25, 0.75], 0.01)
iqr = q3 - q1
limite_inferior = q1 - 1.5 * iqr
limite_superior = q3 + 1.5 * iqr

print(f"Q1={q1:.2f}  Q3={q3:.2f}  IQR={iqr:.2f}")
print(f"Limite inferior: {limite_inferior:.2f}  |  Limite superior: {limite_superior:.2f}")

df_tratado = df_tratado.filter(
    (col("temperatura") >= limite_inferior) & (col("temperatura") <= limite_superior)
)

print(f"Linhas após remoção de outliers: {df_tratado.count():,}")

## 14. Verificação de valores nulos após os tratamentos

Após as etapas de limpeza, é feita uma nova contagem de valores nulos por coluna.

Essa verificação ajuda a decidir quais colunas serão mantidas, removidas ou imputadas.

In [ ]:
expressoes_nulos = []

for c in df_tratado.columns:
    expressoes_nulos.append(
        count(
            when(col(c).isNull(), 1)
        ).alias(c)
    )

df_nulos = df_tratado.select(*expressoes_nulos)

df_nulos.show(truncate=False)

## 15. Percentual de valores nulos por coluna

Além da quantidade absoluta de nulos, também é calculado o percentual de valores ausentes em cada coluna.

Esse percentual será usado para remover colunas com excesso de dados faltantes.

In [ ]:
total_linhas = df_tratado.count()

expressoes_percentual_nulos = []

for c in df_tratado.columns:
    expressoes_percentual_nulos.append(
        round(
            (count(when(col(c).isNull(), 1)) / total_linhas) * 100,
            2
        ).alias(c)
    )

df_percentual_nulos = df_tratado.select(*expressoes_percentual_nulos)

df_percentual_nulos.show(truncate=False)

## 16. Remoção de colunas com muitos valores nulos

Colunas com grande percentual de valores ausentes podem prejudicar os modelos ou exigir imputações pouco confiáveis.

Neste notebook, são removidas as colunas com mais de 40% de valores nulos.

In [ ]:
limite_nulos = 40.0

percentuais = df_percentual_nulos.collect()[0].asDict()

colunas_remover_por_nulos = [
    coluna for coluna, percentual in percentuais.items()
    if percentual is not None and percentual > limite_nulos
]

print("Colunas removidas por excesso de nulos:")
print(colunas_remover_por_nulos)

df_tratado = df_tratado.drop(*colunas_remover_por_nulos)

print("Colunas restantes:", len(df_tratado.columns))

## 17. Imputação de valores ausentes nas variáveis numéricas

Para evitar perda excessiva de dados, os valores nulos nas variáveis numéricas explicativas são preenchidos com a mediana.

A mediana é menos sensível a valores extremos do que a média, por isso é uma escolha adequada para dados meteorológicos.

In [ ]:
colunas_numericas_tratadas = [
    c for c in colunas_numericas
    if c in df_tratado.columns and c != "temperatura"
]

print("Features para imputação:", colunas_numericas_tratadas)

In [ ]:
medianas = {}

for c in colunas_numericas_tratadas:
    valores = df_tratado.select(c).dropna().approxQuantile(c, [0.5], 0.01)
    if len(valores) > 0:
        medianas[c] = valores[0]

print("Medianas calculadas:", medianas)

df_imputado = df_tratado

for c, mediana in medianas.items():
    df_imputado = df_imputado.withColumn(
        c,
        when(col(c).isNull(), mediana).otherwise(col(c))
    )

df_imputado.show(5, truncate=False)

## 18. Remoção de registros sem variáveis temporais

As variáveis temporais são importantes para o objetivo do projeto, pois a proposta envolve analisar variações de temperatura ao longo dos anos.

Por isso, registros sem `ano`, `mes` ou `hora_num` são removidos.

In [ ]:
df_final = df_imputado.filter(
    col("ano").isNotNull() &
    col("mes").isNotNull() &
    col("hora_num").isNotNull()
)

print("Linhas após remover registros sem variáveis temporais:", df_final.count())

## 19. Seleção da base final para modelagem

Nesta etapa são selecionadas as variáveis que serão usadas como entrada para os modelos.

A variável `temperatura` será a saída esperada, ou seja, a variável que os modelos irão tentar prever.

In [ ]:
features_base = [
    "ano", "mes", "dia", "hora_num",
    "latitude", "longitude", "height",
    "umidade", "umidade_maxima", "umidade_minima",
    "pressao", "pressao_maxima", "pressao_minima",
    "precipitacao", "radiacao",
    "velocidade_vento", "rajada_vento", "direcao_vento"
]

features_base = [c for c in features_base if c in df_final.columns]

coluna_alvo = "temperatura"

df_modelagem_base = df_final.select(*(features_base + [coluna_alvo])).dropna()

df_modelagem_base.show(10, truncate=False)
df_modelagem_base.printSchema()
print("Features finais:", features_base)
print(f"Total de linhas para modelagem: {df_modelagem_base.count():,}")

## 20. Separação temporal entre treino e teste

Como o dataset possui natureza temporal, a separação entre treino e teste deve respeitar a ordem cronológica dos registros.

Em vez de utilizar divisão aleatória, os dados são separados por ano (80% dos anos mais antigos para treino; 20% mais recentes para teste). Dessa forma, o modelo é treinado com registros de anos anteriores e avaliado em anos posteriores, simulando um cenário realista e evitando vazamento temporal.

In [ ]:
anos_disponiveis = [
    row["ano"]
    for row in df_modelagem_base.select("ano")
        .distinct()
        .dropna()
        .orderBy("ano")
        .collect()
]

qtd_anos = len(anos_disponiveis)
qtd_anos_teste = builtins.max(1, int(qtd_anos * 0.2))

anos_teste = anos_disponiveis[-qtd_anos_teste:]
anos_treino = anos_disponiveis[:-qtd_anos_teste]

print("Anos disponíveis:", anos_disponiveis)
print("Anos de treino:", anos_treino)
print("Anos de teste:", anos_teste)

df_treino = df_modelagem_base.filter(col("ano").isin(anos_treino))
df_teste = df_modelagem_base.filter(col("ano").isin(anos_teste))

train_path = "/home/jovyan/work/data/processed/weather_sp_train"
test_path = "/home/jovyan/work/data/processed/weather_sp_test"

df_treino.write.mode("overwrite").parquet(train_path)
df_teste.write.mode("overwrite").parquet(test_path)

print(f"\nTreino: {df_treino.count():,} linhas → {train_path}")
print(f"Teste : {df_teste.count():,} linhas → {test_path}")

## 21. Estatísticas finais da base pré-processada

Antes de salvar a base final, são calculadas estatísticas básicas da variável-alvo após todos os tratamentos.

Essa etapa permite verificar se a temperatura permaneceu dentro de uma faixa coerente.

In [ ]:
df_modelagem_base.select(
    count("*").alias("total_registros"),
    round(avg("temperatura"), 2).alias("media_temperatura"),
    round(min("temperatura"), 2).alias("min_temperatura"),
    round(max("temperatura"), 2).alias("max_temperatura"),
    round(stddev("temperatura"), 2).alias("desvio_temperatura")
).show(truncate=False)

## 22. Salvamento da base pré-processada completa

A base final é salva em formato Parquet para ser reutilizada nos próximos notebooks de modelagem.

Além da base completa, também são salvas as bases de treino e teste. Dessa forma, todos os modelos utilizarão exatamente a mesma divisão dos dados.

In [ ]:
output_path = "/home/jovyan/work/data/processed/weather_sp_preprocessado"

df_modelagem_base.write.mode("overwrite").parquet(output_path)

print(f"Base pré-processada completa salva em: {output_path}")

## 23. Teste e verificação da leitura

Por fim, as bases salvas são lidas novamente para confirmar que o processo de gravação ocorreu corretamente.

In [ ]:
df_base_salva = spark.read.parquet("/home/jovyan/work/data/processed/weather_sp_preprocessado")
df_treino_check = spark.read.parquet("/home/jovyan/work/data/processed/weather_sp_train")
df_teste_check  = spark.read.parquet("/home/jovyan/work/data/processed/weather_sp_test")

print("Base completa:")
df_base_salva.show(5, truncate=False)
print("Linhas:", df_base_salva.count())

print(f"Base de treino: {df_treino_check.count():,} linhas")
print(f"Base de teste : {df_teste_check.count():,} linhas")

## Conclusão do Pré-processamento

Neste notebook, a base meteorológica passou por todas as etapas vitais de limpeza, tratamento de valores faltantes e criação de features temporais.

A base está pronta para modelagem. Próximos passos sugeridos:
- **04_modelo_random_forest.ipynb** — RandomForestRegressor
- **05_modelo_linear_regression.ipynb** — LinearRegression com StandardScaler
- **07_comparacao_final.ipynb** — Comparação das métricas e gráficos